# Foundations of Data Science
## S1 Week 03: Data representation I: Matplotlib

**Learning outcomes:** 
In this lab you will learn different plotting methods using Matplotlib. By the end of the lab you should be able to:
- understand how Matplotlib works under the hood,
- draw line plots, stacked plots, bar charts and scatter plots,
- set parameters of your plots.

**Remarks**
- For some exercises you might need to use the Matplotlib documentation or a search engine - the [Tutorials](https://matplotlib.org/3.7.2/tutorials/index.html) and [API reference](https://matplotlib.org/3.7.2/api/index.html) are the most useful sections. It is important to get familiar with looking up how to solve problems online. Throughout your career, you will encounter many problems that other people have encountered too; there is no need to reinvent the wheel.
- We will not provide all the code in the tutorials, but expect you to be able to draw from previous labs to fill in gaps.
- Try not to use copy+paste when coding these labs, as typing will help you memorize the code better.
- Try to understand each detail in the code we provide, and read the comments!

After answering some questions about life expectancy, we will again take the role of a political advisor for the UN, and discuss climate data. 

**Research question:** Who is responsible for the CO<sub>2</sub> emissions world-wide? 

**Data information:** We will use data from [ourworldindata.org](https://ourworldindata.org/co2-and-other-greenhouse-gas-emissions), which we preprocessed. It contains $CO_2$ emissions data from 1900-2017, for different continents, and countries.

## A Matplotlib introduction

Matplotlib is a python plotting library with a similar interface to MATLAB's plot engine. The library is fully compatible with NumPy, and by extension pandas.

Drawing basic graphs with one-liners is quite easy with Matplotlib. However, the Matplotlib documentation or [stackoverflow questions with Matplotlib tags](https://stackoverflow.com/questions/tagged/matplotlib) can feel overwhelming. With over 70,000 lines it is a huge library, and while there are many quite extensive [resources](http://matplotlib.org/resources/index.html); some might be outdated, as this library evolves quickly. As with every lab in this course, we won't teach you the whole library; the goal is for you to get a solid understanding of how the library works, such that if you need to look something up you can understand it quickly. A helpful collection of example figures, in case you are looking for a particular feature you don't know or don't remember, can be found [here](https://matplotlib.org/tutorials/introductory/sample_plots.html).

In this lab we will only work through the main library `matplotlib.pyplot`. Let's start!

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

### A.1 Matplotlib object hierarchy

Let's begin by understanding the structure of a plot and its terminology, which confuses many beginners. Understanding this structure is important, and the lack of understanding stops many developers from drawing good graphs. Copy+Pasting Stack Overflow answers will only get you so far, and if you haven't understood the difference between *axes* and *axis*, it will take you a long time to draw the right graph by trial and error.

A plot object in Matplotlib has a hierarchical (tree-like) structure. At the "root" is the `figure` container.
Each figure can contain multiple `axes` objects, which is the next smallest object in the hierarchy. *An* axes<sup>1</sup> translates into what we think of as an individual plot or graph. This is not to be confused with `axis` (in 2D setting `x` and `y`, and in 3D plots `z`).

<sup>1</sup> In English *axes* is a plural, but in Matplotlib terminology it is singular.

![title](images/fig_map.png)
From the [Matplotlib documentation](https://matplotlib.org/1.5.1/_images/fig_map.png).

In [ ]:
# Let's double check the statements above about the hierarchy
fig, _ = plt.subplots() # Creates a figure with several plots
print(type(fig))
print(type(fig.axes[0])) # as stated above each figure can have several axes, thus type(fig.axes) returns "List"

Below the axes in the hierarchy we find many objects such as legends, labels, lines... Each element of a plot is its own manipulable Python object, all the way down to the ticks and labels.

![title](images/sphx_glr_anatomy_001.png)
From the [Matplotlib documentation](https://matplotlib.org/3.2.1/_images/sphx_glr_anatomy_001.png).

In [ ]:
# Let's double check the hierarchy, and the statement, that we can traverse the tree down all the way to ticks
first_tick = fig.axes[0].yaxis.get_major_ticks()[0]
type(first_tick)

## B Line plots

The default plots in Matplotlib are line plots. They connect different points specified as `([x1, x2, x3, ...], [y1, y2, y3 ...])`. If only one array is given, Matplotlib assumes that the x coordinates are `[0, 1, 2, ...]`.

In [ ]:
plt.plot([1, 2, 3, 4]) # Remark that the coordinates are not (1,1), (2,2)... but (0,1), (1,2), ...
plt.show()

In [ ]:
plt.plot([1, 2, 3, 4], [1, 4, 9, 16])
plt.show()

**Remark** We call these plots "line plots", but we actually plot points that happen to be connected *by default* with straight lines. This can be changed by adding a third parameter, which specifies the colour of the points and how they should be connected. The default is `'b-'`, which stands for *blue* (`b`) *line* (`-`). Let's try some different options.

In [ ]:
# Evenly sampled time at 200ms (0.2s) intervals for 5 seconds
x = np.arange(0., 5., 0.2)
y1 = x
y2 = x**2
y3 = x**3

# Red dashes, blue squares and green triangles
# Note that we can plot multiple sets of data using extra arguments to the plot command
plt.plot(x, y1, 'r--', x, y2, 'bs', x, y3, 'g^')
# A legend is helpful when having several plots in one figure and we can use LaTeX notation for maths
# between dollar signs
plt.legend(['$f(x) = x$', '$f(x) = x^2$', '$f(x) = x^3$'])
plt.show()

### Representing functions

How do we draw mathematical functions, if we can only plot points and straight lines? The plot above already hints at the solution. We just 'sample' a function often enough and keep the distance between two points so small that to the naked eye it looks like a smooth curve.

In [ ]:
def f(t):
    return np.exp(-t) * np.cos(2*np.pi*t)

t1 = np.arange(0.0, 5.0, 0.1)
t2 = np.arange(0.0, 5.0, 0.02)

plt.plot(t1, f(t1), 'bo', t2, f(t2), 'k') # Note that 'k' stands for black
plt.xlabel('$t$') # You can use Latex to write mathematical formulas by using $formula$ 
plt.ylabel('$f(t)=exp(-t) \cdot cos(2\pi t)$')
plt.show() # Remark: t2 samples 5 times more often and thus appears to be a smooth curve

## C Stateful vs Stateless

Before we get into more types of plots, this is a good moment to discuss one last theoretical technicality: the difference between the **stateful** (state-machine) and **stateless** (object-oriented) interfaces.

Above, we said that each figure is an object with different axes, each being objects themselves. However, we created the axes using `plt.plot()`, which is a *function* call not a method. I.e. `plt` refers to the library; it is not an object with a `plot()` member function. But how does the machine know which figure or plot we refer to, when we want to change, for example, the x-axis label?

In the Matplotlib docs we find the following statement:

    “[With pyplot], simple functions are used to add plot elements (lines, images, text, etc.) to the current axes in the current figure.”
    
The above means that almost all functions from pyplot (which we have imported as `plt`, by convention) are implicitly either referring to an existing current figure and axes, or creating them from scratch if none exist. This is the stateful approach.

Modifying the underlying objects directly is the stateless/object-oriented approach. We can do this by calling *methods* of an axes object, which is the object that represents a plot itself.

So `plt.plot()` actually first calls `ax = plt.gca()` (get current axes), and then calls on the current axes `ax.plot()`. This is why many functions that you can call are just wrapper functions, which have either the same name or setter and getter function names, e.g.
- `plt.title()` $\implies$ `ax.set_title()`
- `plt.xlabels()` $\implies$ `ax.set_xlabel()`
- `plt.legend()` $\implies$ `ax.legend()`
- ...

Now that we know the different approaches (stateful and stateless), we can jump between axes and assign properties to different axes. This allows us to create subfigures.

## D Subfigures

We already briefly used `fig, ax = plt.subplots(nrows, ncols)`, where `ax` is a list of axes. By default `subplots()` returns a figure with one axes (nrows=1, ncols=1). Let's draw a figure with several rows and columns.

In [ ]:
fig, ((ax1, ax2, ax3, ax4, ax5),(ax6, ax7, ax8, ax9, ax10)) = plt.subplots(2, 5) 

You could also use `fig, ax = plt.subplots(2, 5)` instead, and then either **unpack** it with `ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9, ax10 = ax.flatten()` or access each axes with **indexing**, i.e. `ax[0][0]`. 

In [ ]:
fig, ax = plt.subplots(2, 5)
ax[0][0].plot([1,2])

This looks cramped, with numbers overlapping. If we're in the **data exploration** phase of the data science process, where the audience for the figure is you or your colleagues, a quick way of preventing the numbers overlapping is to increase the **size of the figure** by setting the `figsize=(width,height)` argument of `plt.subplots()`, as shown later.

However, increasing the figure size could mean you break [FDS Visualisation Principle 4](https://opencourse.inf.ed.ac.uk/inf2-fds/resource-list) "Make the data accessible" because of the effective font size becoming too small. When **communicating data**, it is important to make visualisations accessible to your audience. Reflecting this importance, in FDS we award points for accessible visualisations, and deduct points for inaccessible ones. We give some advice for creating accessible visualisations for data communication in PDF documents and screens below.

### D.1 How to make visualisations with an accessible font size in a printed document of PDF

Although we print documents less than we used to, the PDF document with an A4 page size (297mm high by 210mm wide) is a very common format, e.g. for reports in industry, academic papers and university assignments. Although your audience can zoom into a PDF to make fonts bigger, it can distract from reading the document, and some of the impact of your visualisation will be lost. A good rule thumb is to make your fonts large enough so that they are legible when printed out on paper.

This [PDF file shows you how font sizes look relative to the page size](fontsize-demo.pdf).  

To get a legible font size when communicating data with a printed or PDF document:
- Create the figure with the same dimensions as you want on the page. E.g. if you want a figure that is 6 inches wide by 4 high, create it with `figsize=(6, 4)` to `plt.figure()` or `plot.subplot()`.
- If including in LaTeX document, you can save the file as PDF (e.g. `plt.savefig(myfile.pdf)`). This is a vector format, so does not look grainy. However, if your plot has many points, the PDF can be slow to render.
- In other situations, save to a PNG or JPG files (e.g. `plt.savefig(myfile.png)`). You should set the dpi (dots per inch) to 150, to give a better quality image. You can specify the `dpi` as an argument to `plt.figure()`.
- If text doesn't fit easily on the plot, you can try:
  - rotating labels so they don't overlap
  - `plt.tight_layout()` to automatically adjust the plot​
- If that doesn't work, you can set the font size to 8pt using
```
plt.rcParams['font.size'] = 8
```

### D.2 How not to make visualisations accessible in a printed document or PDFs

If your PDF is A4 size, **do not** use something like `figsize=(12, 6)` and then shrink the resulting file when you include it a document. An A4 document typically has 1in (2.54cm) margins, giving a text width of 6in. Scaling down an image that was 12in wide to 6in wide means scaling down by a factor of 2. Fonts that would be 10pt in size when printed on a large enough bit of paper are now half the size, i.e. 5pt.

This [PDF file gives a demo of font sizes and shrinking images to fit](fontsize-demo.pdf).

### D.3 Making visualisations accessible for a presentation

When you prepare a presentation for a screen (e.g. to include in a Powerpoint presentation) there are few rules of thumb:
- If you create the figure as for a PDF document (see above), i.e. size 6in by 4in with a 10pt font (or possibly 9pt or 8pt font), and include the image so that it fills most of the slide. Doing this will mean the text should be legible from the back of a large lecture hall.
- You should make sure the resolution (DPI) is high enough so that the image doesn't look grainy.

### D.4 Making visualisations look a helpful size in a Jupyter Notebook

Even a small laptop screen is typically wider than an A4 page, so restricting the width to 6in isn't needed for legibility. Some of the advice above to set a high DPI will make your visualisations look too big in Jupyter, because the width of a figure in Jupyter depends on how wide the image is in pixels. By default, to convert from inches to pixels, Matplotlib uses a conversion factor of 72DPI (dots per inch). The default figure size of 6in x 4in corresponds to 432 pixels wide and 288 high, when multiplied by 72dpi. We can see this when we save the last figure to a PNG file like this:

In [ ]:
print("Size in inches", fig.get_size_inches())
print("DPI", fig.get_dpi())
print("Size in pixels", fig.get_dpi()*fig.get_size_inches())
fig.savefig("myfig.png")

If you open `myfig.png` in an image viewer (or your web browser), you should see that its size is 432x288 pixels.

### D.5 Getting the numbers to fit in by changing the figsize in exploratory data analysis

In Matplotlib the size of the figure is specified in inches, i.e. `figsize=(width in inches, heigh inches)`. (An inch is 2.54cm), and can set via the `figsize` argument of `subplots` and `figure`. When doing exploratory work, a quick fix for getting the numbers to fit in is to make the figure bigger, so we can fit more characters in.

We'll try making the figure twice as wide.

In [ ]:
fig, ((ax1, ax2, ax3, ax4, ax5),(ax6, ax7, ax8, ax9, ax10)) = plt.subplots(2, 5, figsize=(12, 4))

The text is still spread out, but overlapping a bit. If you have a small screen, the right hand side of the figure maybe touching the right margin. With a larger screen there is probably still some space. What happens if we make the figure wider than the screen?

In [ ]:
fig, ((ax1, ax2, ax3, ax4, ax5),(ax6, ax7, ax8, ax9, ax10)) = plt.subplots(2, 5, figsize=(30, 4))

The image file is now 2.5 times as wide in terms of pixels, but it has been scaled down to the width of the document on *your* screen. The result is that the labels are now much further apart, and therefore not overlapping each other. However, they are also a lot smaller - about half the size. By default, Matplotlib produces plot labels in 10pt. Now, the labels are about 5pt large. Can you read them? (They are too small for at least one of your lecturers!)

Scaling labels down in size might be OK in data exploration - though you might miss something important if you can't read the scale or labels. For the final visualisation, the labels **must** be legible - at least 8pt. If the plot labels are overlapping you'll need to find some other solution to prevent this happening, e.g. fewer plots, a different grid arrangement or not repeating aligned axes. To make sure you create legible plots **in Jupyter Notebooks but not PDFs**, keep your plots a maximum of 15 inches wide.

The easiest solution in our case, might be to use 5 rows and 2 columns for the figures instead of 2 rows and 5 columns. 

**Exercise 01:** 
- Create a new figure with 10 subplots, with 2 subplots in each row. Make the total width of the figure 10 inches wide and the total height 15 inches.
- Assign the different 10 axes the names ax1, ..., ax10

In [ ]:
# Your code
fig, ((ax1, ax2), (ax3, ax4), (ax5,ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, figsize=(10, 15))

That was quite an in-depth digression on figure sizing. However, once you have understood how the figure sizing works, and how the size of the labels might change, you can easily fix the figure to the right size with the right label size without having to play around for ages.

Now we're going to fill the axes above with data about worldwide CO2 emissions.

The first step towards understanding who is responsible for worldwide CO2 emissions is to look at historical data. We will therefore look at the CO2 emissions of various regions since 1900.

**Exercise 02:** 

a) Load the following csv files as DataFrames
- annual_co2_africa.csv
- annual_co2_europe.csv
- annual_co2_asia.csv
- annual_co2_americas.csv
- annual_co2_china.csv
- annual_co2_eu28.csv
- annual_co2_india.csv
- annual_co2_international.csv
- annual_co2_middleeast.csv
- annual_co2_us.csv

The files are found in `datasets`. Make sure that the DataFrame variable names have the name of the files for the rest of the tutorial to work, i.e. `annual_co2_africa.csv` $\implies$ annual_co2_africa .

b) Finally, once all the files are loaded look at the last few entries of one data frame, to get an idea of the data.

In [ ]:
# Your code
annual_co2_africa_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_africa.csv')
annual_co2_africa = pd.read_csv(annual_co2_africa_loc)
annual_co2_europe_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_europe.csv')
annual_co2_europe = pd.read_csv(annual_co2_europe_loc)
annual_co2_asia_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_asia.csv')
annual_co2_asia = pd.read_csv(annual_co2_asia_loc)
annual_co2_americas_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_americas.csv')
annual_co2_americas = pd.read_csv(annual_co2_americas_loc)
annual_co2_china_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_china.csv')
annual_co2_china = pd.read_csv(annual_co2_china_loc)
annual_co2_eu28_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_eu28.csv')
annual_co2_eu28 = pd.read_csv(annual_co2_eu28_loc)
annual_co2_india_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_india.csv')
annual_co2_india = pd.read_csv(annual_co2_india_loc)
annual_co2_international_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_international.csv')
annual_co2_international = pd.read_csv(annual_co2_international_loc)
annual_co2_middleeast_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_middleeast.csv')
annual_co2_middleeast = pd.read_csv(annual_co2_middleeast_loc)
annual_co2_us_loc = os.path.join(os.getcwd(), 'datasets', 'annual_co2_us.csv')
annual_co2_us = pd.read_csv(annual_co2_us_loc)
annual_co2_china.tail()

Now that we have the data loaded, let's plot the data using subplots, to compare the CO2 emissions across regions.

In [ ]:
# Figure
fig, ax = plt.subplots(5,2, figsize=(6, 10))
ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9, ax10 = ax.flatten()
#Setting titles of subplots
ax1.set_title('Africa')
ax2.set_title('Asia')
ax3.set_title('Americas (other)')
ax4.set_title('EU 28')
ax5.set_title('Europe (other)')
ax6.set_title('China')
ax7.set_title('India')
ax8.set_title('US')
ax9.set_title('Middle East')
ax10.set_title('International travel')
#Setting title of main plot
fig.suptitle('Annual $CO_2$ emissions (billion tonnes) for different regions between 1900-2017', fontsize=15)
# Plotting the data
# Careful, above we mentioned that the y axis is in billion tonnes, the data however is in tonnes, thus we need to divide the data by 1e9
ax1.plot(annual_co2_africa['Year'], annual_co2_africa['Annual CO₂ emissions (tonnes )']/1e9)
ax2.plot(annual_co2_asia['Year'], annual_co2_asia['Annual CO₂ emissions (tonnes )']/1e9)
ax3.plot(annual_co2_americas['Year'], annual_co2_americas['Annual CO₂ emissions (tonnes )']/1e9)
ax4.plot(annual_co2_eu28['Year'], annual_co2_eu28['Annual CO₂ emissions (tonnes )']/1e9)
ax5.plot(annual_co2_europe['Year'], annual_co2_europe['Annual CO₂ emissions (tonnes )']/1e9)
ax6.plot(annual_co2_china['Year'], annual_co2_china['Annual CO₂ emissions (tonnes )']/1e9)
ax7.plot(annual_co2_india['Year'], annual_co2_india['Annual CO₂ emissions (tonnes )']/1e9)
ax8.plot(annual_co2_us['Year'], annual_co2_us['Annual CO₂ emissions (tonnes )']/1e9)
ax9.plot(annual_co2_middleeast['Year'], annual_co2_middleeast['Annual CO₂ emissions (tonnes )']/1e9)
ax10.plot(annual_co2_international['Year'], annual_co2_international['Annual CO₂ emissions (tonnes )']/1e9)
plt.tight_layout()

At first glance, it looks like each region has about the same trajectory. However, if you look at the different y-axes, you will realize that all of them have different scales. This is very important and a problem often occurring in data comparisons. Let's fix it.

In [ ]:
# Global settings (applying to all subplots) .setp(axes, parameter = value, ...)
# You can also just set the parameters for a subplot, e.g. plt.setp(ax1, parameter = value, ...)
# By setting ylim for all axes in ax, we can guarantee the same scale. 
plt.setp(ax, xlim=[1900, 2017], ylim=[0, 10], ylabel = '$CO_2$ emmisions in billion tonnes')
fig # This line is just for Jupyter notebooks, in order to show the figure


Looking at the plots, we notice a few things: 
- China is currently producing far more CO2 emissions than any other region.
- Looking at EU-28 and US data, one can clearly see the 1929 Economic crisis, WW II, 1979 energy crisis and other historical events.
- Even with the CO2 emission reduction efforts of the last few years, we are still far away from the goal of zero CO2 emissions.

**Exercise 03:** 

Make a new plot containing all the different time courses in one single plot. Remember to use a legend for the different time courses, and to set the label for the axis, if necessary. Use a different color for each region, such that the reader can differentiate the different plots. You can use [ColorBrewer](https://colorbrewer2.org/#type=sequential&scheme=YlGnBu&n=9) for example to get different colours. Remember that the reader might have difficulties differentiating different colours.

In [ ]:
# Your code
plt.plot(annual_co2_africa['Year'], annual_co2_africa['Annual CO₂ emissions (tonnes )']/1e9, '#543005')
plt.plot(annual_co2_asia['Year'], annual_co2_asia['Annual CO₂ emissions (tonnes )']/1e9, '#8c510a')
plt.plot(annual_co2_americas['Year'], annual_co2_americas['Annual CO₂ emissions (tonnes )']/1e9, '#bf812d')
plt.plot(annual_co2_eu28['Year'], annual_co2_eu28['Annual CO₂ emissions (tonnes )']/1e9, '#dfc27d')
plt.plot(annual_co2_europe['Year'], annual_co2_europe['Annual CO₂ emissions (tonnes )']/1e9, '#f6e8c3')
plt.plot(annual_co2_china['Year'], annual_co2_china['Annual CO₂ emissions (tonnes )']/1e9, '#c7eae5')
plt.plot(annual_co2_india['Year'], annual_co2_india['Annual CO₂ emissions (tonnes )']/1e9, '#80cdc1')
plt.plot(annual_co2_us['Year'], annual_co2_us['Annual CO₂ emissions (tonnes )']/1e9, '#35978f')
plt.plot(annual_co2_middleeast['Year'], annual_co2_middleeast['Annual CO₂ emissions (tonnes )']/1e9, '#01665e')
plt.plot(annual_co2_international['Year'], annual_co2_international['Annual CO₂ emissions (tonnes )']/1e9, '#003c30')
plt.ylabel('Annual $CO_2$ emissions (billion tonnes)')
plt.title('Annual CO emissions')
plt.legend(['Africa', 'Asia','Americas', 'EU28', 'Europe(other)', 'China', 'India', 'US', 'Middle East', 'International Transport'])
plt.show()
## Note: this looks really small in the notebook - but it would look fine if saved using plt.savefig('annual-co2-emissions.pdf')

**Discussion:** Discuss with your lab partner what the advantages and disadvantages are between the two representations, i.e. a single plot for all regions vs 10 subplots for each region. Write down your answer:

Your answer:
- Cluttered plots difficult to read, easier to directly compare two different time courses.
- Finding 10 different colours for colourblind people is a difficult task.

## E Stack plots, bar charts and  scatter plots

Now that we have seen how the CO2 emissions evolved in the different regions, it might be interesting to see how the CO2 emission evolved worldwide, and which regions have the biggest share over time. We can do this using **stack plots**.

In [ ]:
plt.stackplot(annual_co2_africa['Year'], # x value
              annual_co2_africa['Annual CO₂ emissions (tonnes )']/1e9, # y1 value  
              annual_co2_asia['Annual CO₂ emissions (tonnes )']/1e9,  # y2 value ...
              annual_co2_americas['Annual CO₂ emissions (tonnes )']/1e9, 
              annual_co2_eu28['Annual CO₂ emissions (tonnes )']/1e9, 
              annual_co2_europe['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_china['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_india['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_us['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_middleeast['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_international['Annual CO₂ emissions (tonnes )']/1e9)
plt.legend(['Africa', 'Asia','Americas', 'EU28', 'Europe(other)', 'China', 'India', 'US', 'Middle East', 'International Transport'])
plt.show()

**Discussion** Discuss with your lab partner what the advantages and disadvantages are of a stack plot and a line plot, and when to us which format. Discuss with your partner what the problems are of the plot above. Write down your answer:

Your answer:

Line plots: 
- Helpful to see more details and events, such as e.g. Great Depression,
- a comparison of two plots is easier.

Stack plot:
- Easier to see which trajectory has the biggest impact along x-axis.

Problems:
- Legend is on top of the plot
- Colourblind people will have difficulties with the red and green part of the plot.

We have seen that currently China has the biggest CO2 emission, but what about the cumulative CO2 emissions from 1900 to 2017?

### Categorical data representation

**Exercise 04:** 

a) Load the `cumulative_co2_emissions.csv` file, from `datasets`, and save it as a DataFrame with the variable name `cumulative_co2_emissions`.

b) Check how many entries the file has.

c) Use `.head(length)` to show all data entries.

In [ ]:
# Your code
cumulative_co2_emissions_loc = os.path.join(os.getcwd(), 'datasets', 'cumulative_co2_emissions.csv')
cumulative_co2_emissions = pd.read_csv(cumulative_co2_emissions_loc)
size = len(cumulative_co2_emissions)
cumulative_co2_emissions.head(size)

The entity is a categorical variable and the cumulative emissions are numeric. It's technically possible to represent such data either with line plots, where on the x-axis we use an array of categories, and on the y-axis the values, or we can use **bar charts**. Let's compare how the three work for this data.

In [ ]:
region = cumulative_co2_emissions['Entity']
co2_emissions = cumulative_co2_emissions['Cumulative CO₂ emissions by region (billion tonnes)']
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 5))
ax1.plot(region, co2_emissions, 'bo')
ax2.plot(region, co2_emissions, 'bo-')
ax3.bar(region, co2_emissions)

**Discussion** Discuss with your lab partner which representation is the most suitable for the above data. Also can you spot any problems common to all the plots?

Your answer:
- Bar chart is the best representation of the three, since there is no natural connection between or ordering of the categories
- Y axis label is missing
- X tick labels are overlapping
- Font size is too small
- Y scale doesn't start at 0 on left & middle plots

A typical problem, seen in the plots above, when using categorical data, is that the names of the categories are often too long and overlap. One way to solve it would be to swap the x and y axes.

In [ ]:
plt.barh(region, co2_emissions) # Careful: in .barh() the y-axis is the first parameter

Another option is to incline the labels of the categorical data:

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 3))
ax1.bar(region, co2_emissions)
ax2.bar(region, co2_emissions)
ax3.bar(region, co2_emissions)
ax1.set_xticks(region)
ax2.set_xticks(region)
ax3.set_xticks(region)
ax1.set_xticklabels(region, rotation=45)
ax2.set_xticklabels(region, rotation=45, ha='left')
ax3.set_xticklabels(region, rotation=45, ha='right')

`ha` stands for horizontal alignment, and determines the alignment of the text with the ticks. If you compare the three plots, you will see that without any `ha` the text will be aligned such that the middle of the word coincides with the tick. `ha='left'` leads to an alignment with the left end of the word and the tick and `ha='right'` leads to the right end of the word being aligned with the tick.

As you can see above, if you compare the output of CO2 emissions across different regions since the industrial revolution, Europe and the US have emitted far more CO2 than any other region. These two regions have emitted together more than the rest of the world combined.

### High dimensional data representation

For data sets with high dimensionality, **scatter plots** are a great option.  Data points are plotted on horizontal and vertical axis in order to show the dependence between variables. In addition to *x* and *y* axes, a third variable can be added represented by the *size* of the markers, thus adding yet another dimension to the plot. Finally, by adding different data sets with different *colours* to the plot, we can add in addition to three continuous a fourth categorical dimension. 

Let's see how the CO2 emissions compare with the GDP of different countries and their respective population size.

**Exercise 05:**

Load the `co-emissions-per-capita-vs-gdp-per-capita-international-.csv` file from `datasets`, and save it as `co2_vs_gdp`. Check out the first few entries. (Remark: we have preprocessed the data to contain only countries with a population >= 20 Million people).

In [ ]:
# Your code
co2_vs_gdp_loc = os.path.join(os.getcwd(), 'datasets', 'co-emissions-per-capita-vs-gdp-per-capita-international-.csv')
co2_vs_gdp = pd.read_csv(co2_vs_gdp_loc)
co2_vs_gdp.head()

In [ ]:
plt.xscale('log') # Note: we are using a log scale here to make the changes in the data more apparent
# Scatter plot, with size of blobs representing population size
plt.scatter(co2_vs_gdp['GDP per capita (international-$ (in 2011 prices))'],
            co2_vs_gdp['CO₂ emissions per capita (tonnes per capita)'],
            s=co2_vs_gdp['Total population (Gapminder)']/1e6,
            c='c',
            #If you don't use edgecolor, many blobs won't be visible. Try it out!
            edgecolors='k')
plt.xlabel('GDP per capita (international-$ (in 2011 prices))')
plt.ylabel('CO₂ emissions per capita (tonnes per capita)')

The above scatter plot is not very helpful. We don't know which countries are represented by which blob, and which regions are at which end of the scale... It is your job now to improve it.

**Exercise 06:**

a) In the last lab, the `drinks_by_country.csv` dataset contained the continents. Load that dataset (we have made a copy in this lab's `dataset` folder).

b) Merge the new DataFrame with the previous DataFrame (`co2_vs_gdp`). Careful: In `co2_vs_gdp` the country column is called `Entity` whereas in `drinks_by_country` the country column is called `Country`.

c) Draw a new scatter plot using colours to represent in which continent the countries lie. Hint: Use a loop to filter the DataFrame for each unique values of the `Continent` column. Add a scatterplot for each newly filtered DataFrame to the current axes, using a different color in each loop. You can use the following `colours` array provided in the code cell below.

d) Create a legend for the colours.

d*) The blobs in the legend all have different sizes. Make them equal sized by saving your legend to a variable `lgnd = plt.legend()` and then call `lgnd.legendHandles[i]._sizes = [50]` with `i` ranging from 0 to the length of the continents array to make the blobs equal sized.

d) Find in your DataFrame:
- the country with the biggest population
- the country with the highest GDP
- the country with the lowest GDP
- the country with the highest CO2 emissions

e) Use Stackoverflow to find out how to add names to data points in scatter plots. Add the country names you have found above to the plot, in addition to one name of your choice from each continent.

In [ ]:
colours = ['#d73027','#fc8d59','#fee090','#e0f3f8','#91bfdb','#4575b4']

# Your code
# a
drinks_by_country_loc = os.path.join(os.getcwd(), 'datasets', 'drinks_by_country.csv')
drinks_by_country = pd.read_csv(drinks_by_country_loc)

In [ ]:
# b
co2_vs_gdp_by_continent = pd.merge(co2_vs_gdp, drinks_by_country, left_on=['Entity'],right_on=['Country'], how='inner')

# You can skip to (c) at this point, since the exercise is really about visualisation.
# However, the rest of this part of the solution to (b) shows you how to deal with
# mismatch between valuexploratoryexploratoryes of categorical variables when merging.

# Note that co2_vs_gdp_by_continent has fewer rows than co2_vs_gdp
print('# rows in drinks_by_country: %d; # rows in co2_vs_gdp: %d; # rows in co2_vs_gdp_by_continent: %d'%(len(drinks_by_country), len(co2_vs_gdp), len(co2_vs_gdp_by_continent)))

In [ ]:
# This implises that the Entity and country names do not all match.
# We can see which ones in co2_vs_gdp, but not in drinks_by_country like this:
set(co2_vs_gdp['Entity']) - set(drinks_by_country['Country'])

In [ ]:
# Now we can see what the full list of country names is in drinks_by_country, and work out
# what we should rename them to so that they have the same name co2_vs_gdp.
', '.join(drinks_by_country['Country'])

In [ ]:
# OK, it looks like we can make the following substitutions and merge again - there still will be on
# country fewer in the co2_vs_gdp_by_continent data frame than in the co2_vs_gdp data frame
drinks_by_country2 = drinks_by_country
drinks_by_country2['Country'].replace({'USA': 'United States',
                                          'DR Congo': 'Democratic Republic of Congo',
                                          'Russian Federation': 'Russia'}, inplace=True)

co2_vs_gdp_by_continent = pd.merge(co2_vs_gdp, drinks_by_country2, left_on=['Entity'],right_on=['Country'], how='inner')
## Note that co2_vs_gdp_by_continent has fewer rows than co2_vs_gdp
print('# rows in drinks_by_country: %d; # rows in co2_vs_gdp: %d; # rows in co2_vs_gdp_by_continent: %d'%(len(drinks_by_country), len(co2_vs_gdp), len(co2_vs_gdp_by_continent)))

In [ ]:
# c
continents = pd.unique(co2_vs_gdp_by_continent['Continent'])

plt.figure(figsize=(9,6))
# Note: to fit this into a report, we'd need plt.figure(figsize=(6,4)), which would mean we'd need to fix
# overlapping labels. plt.tight_layout() could help, a bit, but we might need to change the format of the plot to make it work.

for i in range(len(continents)):
    co2_vs_gdp_by_single_continent = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Continent'] == continents[i]]
    scatter = plt.scatter(co2_vs_gdp_by_single_continent['GDP per capita (international-$ (in 2011 prices))'],
                co2_vs_gdp_by_single_continent['CO₂ emissions per capita (tonnes per capita)'],
                c=colours[i],
                s=co2_vs_gdp_by_single_continent['Total population (Gapminder)']/1e6,
                edgecolors='k')
plt.xlabel('GDP per capita (international-$ (in 2011 prices))')
plt.ylabel('CO₂ emissions per capita (tonnes per capita)')
plt.xscale('log')
# d
lgnd = plt.legend(loc="upper left", labels=continents)
# d*
for i in range(len(continents)):
    lgnd.legend_handles[i]._sizes = [50]

# e
largest_population = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Total population (Gapminder)'] == max(co2_vs_gdp_by_continent['Total population (Gapminder)'])].iloc[0]['Entity']
highest_gdp = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'] == max(co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'])].iloc[0]['Entity']
lowest_gdp  = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'] == min(co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'])].iloc[0]['Entity']
highest_co2 = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['CO₂ emissions per capita (tonnes per capita)'] == max(co2_vs_gdp_by_continent['CO₂ emissions per capita (tonnes per capita)'])].iloc[0]['Entity']

# f
# One country from each continent
countries = [co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Continent'] == continent].iloc[-1]['Entity'] for continent in continents]

countries.append(largest_population)
countries.append(highest_gdp)
countries.append(lowest_gdp)
countries.append(highest_co2)

for i in range(len(countries)):
    plt.annotate(countries[i], # this is the text
                 (co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Entity'] == countries[i]]['GDP per capita (international-$ (in 2011 prices))'],co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Entity'] == countries[i]]['CO₂ emissions per capita (tonnes per capita)']), # this is the point to label
                 ha='center') # horizontal alignment can be left, right or center

**Exercise 07:**

Create a figure with 4 subplots:
- one containing the line plot from Exercise 02, 
- one containing the stack plot
    - move the legend to the top left corner,
- one containing the bar chart with the cumulative CO2 emissions 
    - make the two bars with the highest CO2 emissions red and all the other bars grey,
- and one containing the scatter plot you created in Exercise 05.

Important: Not all plots above contain all the necessary data (e.g. legends, axis, labels, colourblind friendly colours...). Make sure that your figure contains all the necessary information but not too much. Add a title to your figure.

Finally, save your figure into a png file.

It is in your own interest not to copy and paste the previous code but to type the code yourself.

Discuss with your lab partner the data. Here is a highly recommendable [short video](https://www.youtube.com/watch?v=ipVxxxqwBQw) with more information.

In [ ]:
# Your code
fig, ((ax1, ax2),(ax3,ax4)) = plt.subplots(2,2, figsize = (15,15))

# Note: this looks OK in a Jupyter notebook. To put into a report with legible font sizes,
# ideally we'd use plt.figure(figsize=(6,6)), which would mean we'd need 
# to fix problems with overlapping labels and axis names. "plt.tight_layout()" right at the end
# would help a bit. We might also be able to reduce the fontsize to
# 8pt like this:
# import Matplotlib
# matplotlib.\1cParams['font.size'] = 8
# But there would still be other tweaks required to make the plots look really good

# First plot

ax1.plot(annual_co2_africa['Year'], annual_co2_africa['Annual CO₂ emissions (tonnes )']/1e9, '#543005')
ax1.plot(annual_co2_asia['Year'], annual_co2_asia['Annual CO₂ emissions (tonnes )']/1e9, '#8c510a')
ax1.plot(annual_co2_americas['Year'], annual_co2_americas['Annual CO₂ emissions (tonnes )']/1e9, '#bf812d')
ax1.plot(annual_co2_eu28['Year'], annual_co2_eu28['Annual CO₂ emissions (tonnes )']/1e9, '#dfc27d')
ax1.plot(annual_co2_europe['Year'], annual_co2_europe['Annual CO₂ emissions (tonnes )']/1e9, '#f6e8c3')
ax1.plot(annual_co2_china['Year'], annual_co2_china['Annual CO₂ emissions (tonnes )']/1e9, '#c7eae5')
ax1.plot(annual_co2_india['Year'], annual_co2_india['Annual CO₂ emissions (tonnes )']/1e9, '#80cdc1')
ax1.plot(annual_co2_us['Year'], annual_co2_us['Annual CO₂ emissions (tonnes )']/1e9, '#35978f')
ax1.plot(annual_co2_middleeast['Year'], annual_co2_middleeast['Annual CO₂ emissions (tonnes )']/1e9, '#01665e')
ax1.plot(annual_co2_international['Year'], annual_co2_international['Annual CO₂ emissions (tonnes )']/1e9, '#003c30')
ax1.set_xlabel('Year')
ax1.set_ylabel('Annual $CO_2$ emissions (billion tonnes) from 1900-2017')
ax1.set_title('Annual $CO_2$ emissions (billion tonnes) from 1900-2017')
ax1.legend(['Africa', 'Asia','Americas', 'EU28', 'Europe(other)', 'China', 'India', 'US', 'Middle East', 'International Transport'])

# Second plot

ax2.stackplot(annual_co2_africa['Year'], # x value
              annual_co2_africa['Annual CO₂ emissions (tonnes )']/1e9, # y1 value  
              annual_co2_asia['Annual CO₂ emissions (tonnes )']/1e9,  # y2 value ...
              annual_co2_americas['Annual CO₂ emissions (tonnes )']/1e9, 
              annual_co2_eu28['Annual CO₂ emissions (tonnes )']/1e9, 
              annual_co2_europe['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_china['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_india['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_us['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_middleeast['Annual CO₂ emissions (tonnes )']/1e9,
              annual_co2_international['Annual CO₂ emissions (tonnes )']/1e9, colors=['#543005','#8c510a','#bf812d','#dfc27d','#f6e8c3','#c7eae5','#80cdc1','#35978f','#01665e','#003c30'])
ax2.legend(['Africa', 'Asia','Americas', 'EU28', 'Europe(other)', 'China', 'India', 'US', 'Middle East', 'International Transport'], loc='upper left')
ax2.set_xlabel('Year')
ax2.set_title('Annual $CO_2$ emissions from 1900-2017')
ax2.set_ylabel('Annual $CO_2$ emissions (billion tonnes) from 1900-2017')

#Third plot

ax3.bar(region, co2_emissions, color=['grey', 'grey', 'grey', 'grey', 'grey', 'grey', 'grey', 'red', 'red'])
ax3.set_xticks(region)
ax3.set_xticklabels(region, rotation=45, ha='right')
ax3.set_ylabel('Cumulative CO₂ emissions by region (billion tonnes) from 1900-2017')
ax3.set_title('Cumulative $CO_2$ emissions from 1900-2017')

# Fourth plot

for i in range(len(continents)):
    co2_vs_gdp_by_single_continent = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Continent'] == continents[i]]
    scatter = ax4.scatter(co2_vs_gdp_by_single_continent['GDP per capita (international-$ (in 2011 prices))'],
                co2_vs_gdp_by_single_continent['CO₂ emissions per capita (tonnes per capita)'],
                c=colours[i],
                s=co2_vs_gdp_by_single_continent['Total population (Gapminder)']/1e6,
                edgecolors='k')
ax4.set_xlabel('GDP per capita (international-$ (in 2011 prices))')
ax4.set_ylabel('CO₂ emissions per capita (tonnes per capita)')
ax4.set_title('CO₂ emissions per capita')

ax4.set_xscale('log')
lgnd = ax4.legend(loc="upper left", labels=continents)
for i in range(len(continents)):
    lgnd.legendHandles[i]._sizes = [50]

largest_population = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Total population (Gapminder)'] == max(co2_vs_gdp_by_continent['Total population (Gapminder)'])].iloc[0]['Entity']
highest_gdp = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'] == max(co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'])].iloc[0]['Entity']
lowest_gdp  = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'] == min(co2_vs_gdp_by_continent['GDP per capita (international-$ (in 2011 prices))'])].iloc[0]['Entity']
highest_co2 = co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['CO₂ emissions per capita (tonnes per capita)'] == max(co2_vs_gdp_by_continent['CO₂ emissions per capita (tonnes per capita)'])].iloc[0]['Entity']
countries = [co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Continent'] == continent].iloc[-1]['Entity'] for continent in continents]
countries.append(largest_population)
countries.append(highest_gdp)
countries.append(lowest_gdp)
countries.append(highest_co2)

for i in range(len(countries)):
    ax4.annotate(countries[i], # this is the text
                 (co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Entity'] == countries[i]]['GDP per capita (international-$ (in 2011 prices))'],co2_vs_gdp_by_continent[co2_vs_gdp_by_continent['Entity'] == countries[i]]['CO₂ emissions per capita (tonnes per capita)']), # this is the point to label
                 ha='center') # horizontal alignment can be left, right or center
    
fig.savefig("CO2_emissions.png")